# OCR CCCD — Thử từng bước pipeline (Google Colab)

Notebook này chạy thử **từng bước** của pipeline OCR Căn cước công dân, đúng như trong
`backend/ocr.py` của dự án Hospital Wayfinding:

1. **Tiền xử lý** — xoay theo EXIF, thu nhỏ ảnh lớn, tăng tương phản.
2. **Model (PaddleOCR)** — phát hiện vùng chữ (bounding box) + nhận diện nội dung.
3. **Hậu xử lý** — làm sạch các dòng text.
4. **Trích xuất** — lấy các trường: số CCCD, họ tên, ngày sinh, giới tính, quốc tịch, quê quán, nơi thường trú.

> Chạy lần lượt từng cell (Shift+Enter). Cell cài đặt lần đầu mất ~1–2 phút.
> Colab dùng Python ≤ 3.12 nên cài được PaddlePaddle (máy local Python 3.14 thì không).

## 0 · Cài đặt thư viện

In [ ]:
# Cài PaddleOCR + PaddlePaddle (chỉ cần chạy 1 lần mỗi phiên Colab)
!pip install -q paddleocr paddlepaddle pillow

## 1 · Tải ảnh CCCD lên
Chạy cell dưới rồi chọn 1 ảnh CCCD từ máy. (Nếu không dùng Colab, gán thủ công
`IMAGE_PATH = "duong_dan_anh.jpg"`.)

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    IMAGE_PATH = list(uploaded.keys())[0]
except Exception:
    IMAGE_PATH = "cccd.jpg"  # đổi thành đường dẫn ảnh của bạn nếu chạy ngoài Colab
print("Ảnh dùng để test:", IMAGE_PATH)

## 2 · Khai báo chung (hằng số + bỏ dấu tiếng Việt)

In [ ]:
import io, os, re, unicodedata
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

MAX_SIDE = 1600  # giới hạn cạnh dài của ảnh

def strip_accents(text: str) -> str:
    """Bỏ dấu tiếng Việt + chữ thường, ví dụ 'Họ tên' -> 'ho ten'."""
    text = unicodedata.normalize("NFD", text.lower())
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return text.replace("đ", "d")

## 3 · BƯỚC 1 — Tiền xử lý
Xoay ảnh đúng chiều theo EXIF, thu nhỏ nếu quá lớn, tăng tương phản cho chữ rõ hơn.
So sánh ảnh **gốc** và ảnh **sau tiền xử lý**.

In [ ]:
def preprocess_pil(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)   # ảnh điện thoại hay xoay theo EXIF
    img = img.convert("RGB")
    long_side = max(img.size)
    if long_side > MAX_SIDE:             # thu nhỏ ảnh quá to (OCR nhanh hơn)
        scale = MAX_SIDE / long_side
        img = img.resize((int(img.width * scale), int(img.height * scale)))
    img = ImageOps.autocontrast(img)     # kéo giãn tương phản
    return img

print("Ảnh gốc:")
display(Image.open(IMAGE_PATH).convert("RGB"))

pre_img = preprocess_pil(IMAGE_PATH)
print("Sau tiền xử lý:")
display(pre_img)

## 4 · BƯỚC 2 — Model: text detection + recognition
PaddleOCR vừa **phát hiện vùng chữ** (polygon/bounding box) vừa **nhận diện** nội dung
kèm độ tin cậy. Ta vẽ box + đánh số lên ảnh và in danh sách chữ đọc được.

> Lưu ý Colab: ta tắt oneDNN (`FLAGS_use_mkldnn=0` + `enable_mkldnn=False`) để tránh lỗi `NotImplementedError ... onednn_instruction` của PaddlePaddle 3.x trên CPU Colab.

In [ ]:
import os
# Tắt oneDNN/MKLDNN để tránh lỗi "NotImplementedError: ConvertPirAttribute2RuntimeAttribute
# ... onednn_instruction" của PaddlePaddle 3.x trên CPU của Colab. Phải đặt TRƯỚC khi import paddle.
os.environ["FLAGS_use_mkldnn"] = "0"

from paddleocr import PaddleOCR

# Khởi tạo 1 lần (lần đầu sẽ tải model ~vài chục MB). lang="vi" cho tiếng Việt.
# enable_mkldnn=False: chốt thêm việc tắt oneDNN (một số bản PaddleOCR nhận tham số này).
try:
    ocr_engine = PaddleOCR(lang="vi", use_textline_orientation=False, enable_mkldnn=False)
except TypeError:
    ocr_engine = PaddleOCR(lang="vi", use_textline_orientation=False)

pre_img.save("_pre.png")
result = ocr_engine.predict("_pre.png")
res = result[0]

texts  = list(res.get("rec_texts", []))    # nội dung từng vùng
scores = list(res.get("rec_scores", []))   # độ tin cậy 0..1
polys  = list(res.get("rec_polys", [])) or list(res.get("rec_boxes", []))  # toạ độ vùng

def poly_to_points(poly):
    """Đưa polygon 4 điểm hoặc box [x1,y1,x2,y2] về [[x,y],...]."""
    pts = list(poly)
    if len(pts) == 4 and all(not hasattr(p, "__len__") for p in pts):
        x1, y1, x2, y2 = (int(v) for v in pts)
        return [[x1, y1], [x2, y1], [x2, y2], [x1, y2]]
    return [[int(p[0]), int(p[1])] for p in pts]

annotated = pre_img.copy()
draw = ImageDraw.Draw(annotated)
for i, t in enumerate(texts):
    pts = poly_to_points(polys[i]) if i < len(polys) else []
    if pts:
        draw.line([tuple(p) for p in pts] + [tuple(pts[0])], fill=(2, 132, 199), width=3)
        draw.text((pts[0][0], max(0, pts[0][1] - 14)), str(i + 1), fill=(220, 38, 38))

print("Bounding box + số thứ tự vùng chữ:")
display(annotated)

print("\nChữ nhận diện (kèm độ tin cậy):")
for i, t in enumerate(texts):
    sc = scores[i] if i < len(scores) else 0.0
    print(f"#{i+1:2d}  [{sc:.2f}]  {t}")

## 5 · BƯỚC 3 — Hậu xử lý
Bỏ khoảng trắng thừa và dòng rỗng. So sánh **thô** ↔ **đã làm sạch**.

In [ ]:
def postprocess(lines):
    cleaned = []
    for ln in lines:
        ln = re.sub(r"\s+", " ", ln).strip()
        if ln:
            cleaned.append(ln)
    return cleaned

cleaned = postprocess(texts)

print("Thô (từ model):")
for l in texts:
    print("  ", repr(l))
print("\nSau hậu xử lý:")
for l in cleaned:
    print("  ", repr(l))

## 6 · BƯỚC 4 — Trích xuất thông tin (4 trường)

Chỉ lấy 4 trường cần cho đăng ký: **số CCCD, họ tên, ngày sinh, địa chỉ**.

- **Số CCCD / Ngày sinh**: tìm theo *mẫu* — dãy 9/12 chữ số, và ngày `dd/mm/yyyy`.
- **Họ tên / Địa chỉ**: tìm theo *nhãn*. Trên CCCD, nhãn ("Họ và tên / Full name",
  "Nơi thường trú / ...") đứng **một dòng**, **giá trị nằm ở DÒNG NGAY DƯỚI** — nên hàm
  `value_below_label` lấy dòng kế tiếp (hoặc phần sau ':' nếu giá trị nằm cùng dòng).

In [ ]:
_DATE_RE = re.compile(r"\b(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{4})\b")

def value_below_label(lines, keywords):
    """Lấy giá trị 1 trường có nhãn trên CCCD.
    - Tìm dòng chứa nhãn (so khớp KHÔNG DẤU nhờ strip_accents).
    - Giá trị thường nằm ở DÒNG NGAY DƯỚI nhãn; nếu viết 'nhãn: giá trị' thì lấy sau ':'.
    """
    for i, line in enumerate(lines):
        if not any(kw in strip_accents(line) for kw in keywords):
            continue
        if ":" in line:                       # giá trị cùng dòng, sau dấu ':'
            after = line.split(":", 1)[1].strip()
            if after:
                return after
        if i + 1 < len(lines):                # giá trị ở dòng kế tiếp
            return lines[i + 1].strip()
        return ""
    return ""

def extract_fields(lines):
    # Số CCCD: dòng còn đúng 9 (CMND) hoặc 12 (CCCD) chữ số sau khi bỏ ký tự khác.
    id_number = ""
    for line in lines:
        digits = re.sub(r"\D", "", line)
        if len(digits) in (9, 12):
            id_number = digits
            break

    # Ngày sinh: mẫu dd/mm/yyyy đầu tiên.
    dob = ""
    m = _DATE_RE.search("\n".join(lines))
    if m:
        d, mo, y = m.groups()
        dob = f"{int(d):02d}/{int(mo):02d}/{y}"

    # Họ tên & Địa chỉ: giá trị nằm ở dòng dưới nhãn.
    full_name = value_below_label(lines, ["ho va ten", "ho ten", "full name"])
    address = value_below_label(
        lines, ["noi thuong tru", "thuong tru", "noi cu tru", "place of residence", "dia chi"]
    )
    return {"id_number": id_number, "full_name": full_name, "dob": dob, "address": address}

fields = extract_fields(cleaned)
import json as _json
print(_json.dumps(fields, ensure_ascii=False, indent=2))